# Phase 1 — Chunking at the citable unit

**What this corpus is for.** The component detects claims that fall *in scope of*
D.Lgs. 30/2026, so the prevalence of such claims can be compared before and after
27 September 2026. That purpose sets the design:

* **`regime` is the treatment variable.** A provision is either new (binding from
  2026-09-27) or pre-existing. Claims matching pre-existing text were *already*
  unlawful — they are the control arm, and should not move much across the date.
  Claims matching new text are the treated arm. Getting one provision into the
  wrong arm biases the headline result, so `regime` is derived from the document's
  own amendment apparatus and asserted, never typed in.
* **A chunk must be legally self-sufficient**, or the LLM cannot reason with it.
* **Exact letter-level precision matters less than consistency and coverage.**
  A claim routed to Art. 21 c.1 lett. b) instead of Art. 23 lett. d-ter is still
  counted as in-scope; a claim missed entirely is lost from the study.

In [1]:
import os, sys, json, collections, re
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
from ucpd_rag import extract, structure, chunks as K

pd.set_option("display.max_colwidth", 200); pd.set_option("display.width", 220)
text = extract.load()
cs = K.build(text)
problems = K.verify(cs, text)
print("chunks: %d" % len(cs))
print("verification problems:", problems or "none")
K.save(cs)

chunks: 106
verification problems: none


'/sessions/rcw-018waw6qzvfevwdqdgqdgyja/mnt/greenwashing-claim-detector/rag/build/chunks.json'

## 1. Shape of the corpus

Scope is Art. 18–23. **Art. 19 is included** and was not in the original brief: its
comma 3 is the *lex specialis* rule — where EU rules govern specific aspects, they
prevail over this title. Food health and nutrition claims are governed by
Reg. (CE) 1924/2006, so for a large slice of grocery claims Art. 19 is the rule
that says this title may be displaced. A detector that never surfaces it will
over-report in-scope claims.

Excluded, as decisions: **Art. 24–26** (aggressive practices — a claim string from
product copy cannot realise harassment or coercion) and **Art. 45/48/49/65-ter**
(touched by the same decree, but pre-contractual information duties rather than
claims).

In [2]:
df = pd.DataFrame(cs)
piv = df.pivot_table(index=["article", "chunk_type"], columns="regime",
                     values="chunk_id", aggfunc="count", fill_value=0)
piv["total"] = piv.sum(axis=1)
piv

,regime,new-2026,pre-existing,total
article,chunk_type,,,
18,definition,10,15,25
19,scope-limit,0,7,7
20,general-clause,0,6,6
21,general-clause,3,13,16
22,general-clause,1,12,13
23,blacklist,12,27,39


In [3]:
print("chunk_type totals")
print(df.chunk_type.value_counts().to_string())
print()
print("regime totals")
print(df.regime.value_counts().to_string())
print()
print("length of `text` (chars): min %d / median %d / max %d"
      % (df.text.str.len().min(), df.text.str.len().median(), df.text.str.len().max()))

chunk_type totals
chunk_type
blacklist         39
general-clause    35
definition        25
scope-limit        7

regime totals
regime
pre-existing    80
new-2026        26

length of `text` (chars): min 40 / median 239 / max 1255


## 2. The treatment set

These are the provisions D.Lgs. 30/2026 introduces or replaces. Everything else in
the corpus is control. This list is the experiment: if the law bites, claims
matching *these* should thin out after 2026-09-27 while control claims hold steady.

In [4]:
tre = df[df.regime == "new-2026"][["citation_it", "chunk_type", "applicable_from"]]
print("provisions in the treated arm: %d\n" % len(tre))
tre.reset_index(drop=True)

provisions in the treated arm: 26



,citation_it,chunk_type,applicable_from
0,"Art. 18, comma 1, lett. c-bis, Codice del Consumo",definition,2026-09-27
1,"Art. 18, comma 1, lett. n-quater, Codice del Consumo",definition,2026-09-27
2,"Art. 18, comma 1, lett. n-quinquies, Codice del Consumo",definition,2026-09-27
3,"Art. 18, comma 1, lett. n-sexies, Codice del Consumo",definition,2026-09-27
4,"Art. 18, comma 1, lett. n-septies, Codice del Consumo",definition,2026-09-27
5,"Art. 18, comma 1, lett. n-octies, Codice del Consumo",definition,2026-09-27
6,"Art. 18, comma 1, lett. n-novies, Codice del Consumo",definition,2026-09-27
7,"Art. 18, comma 1, lett. n-decies, Codice del Consumo",definition,2026-09-27
8,"Art. 18, comma 1, lett. n-undecies, Codice del Consumo",definition,2026-09-27
9,"Art. 18, comma 1, lett. n-duodecies, Codice del Consumo",definition,2026-09-27


In [5]:
# every treated provision must carry the same date, or the arm is incoherent
assert set(df[df.regime == "new-2026"].applicable_from) == {"2026-09-27"}
assert set(df[df.regime == "new-2026"].source) == {"D.Lgs. 30/2026"}
print("treated arm is coherent: one decree, one date")
print()
print("control arm sources:")
print(df[df.regime == "pre-existing"].source.value_counts().to_string())

treated arm is coherent: one decree, one date

control arm sources:
source
base D.Lgs. 206/2005                          74
amended, marker not attached in source PDF     6


## 3. Awkward cases — before / after

The splitter's first version ended each unit at its **own line**. Three units carry
nested numbered sub-points on separate lines, and all three were silently gutted.
`before` below is that naive behaviour, reproduced rather than described.

In [6]:
def naive_unit(article, letter):
    """Reproduces the first-line-only split, for comparison."""
    a = structure.get_article(article, text)
    body = text[a["body_start"]:a["end"]]
    for m in re.finditer(r"^.+$", body, re.M):
        ln = m.group(0)
        if ln.lstrip("(").startswith(letter + ")"):
            return structure.strip_apparatus(ln)
    return None

by = {c["chunk_id"]: c for c in cs}
cases = [
    ("18", "c-bis",     "letter with nested numbered sub-points"),
    ("18", "n-septies", "nested sub-points AND a cross-reference"),
    ("23", "f",         "blacklist letter with nested sub-points"),
]
rows = []
for art, L, why in cases:
    c = by[K._chunk_id(art, "1", L)]
    b = naive_unit(art, L)
    rows.append({"unit": c["citation_it"], "case": why,
                 "before (chars)": len(b), "after (chars)": len(c["text"]),
                 "regime before": "pre-existing" if "((57))" not in b else "new-2026",
                 "regime after": c["regime"]})
pd.DataFrame(rows)

,unit,case,before (chars),after (chars),regime before,regime after
0,"Art. 18, comma 1, lett. c-bis, Codice del Consumo",letter with nested numbered sub-points,14,506,pre-existing,new-2026
1,"Art. 18, comma 1, lett. n-septies, Codice del Consumo",nested sub-points AND a cross-reference,345,1255,pre-existing,new-2026
2,"Art. 23, comma 1, lett. f, Codice del Consumo",blacklist letter with nested sub-points,80,380,pre-existing,pre-existing


In [7]:
for art, L, why in cases:
    c = by[K._chunk_id(art, "1", L)]
    print("=" * 78)
    print("%s   (%s)" % (c["citation_it"], why))
    print("=" * 78)
    print("BEFORE (%d chars):" % len(naive_unit(art, L)))
    print("   " + naive_unit(art, L))
    print()
    print("AFTER (%d chars):" % len(c["text"]))
    for ln in c["text"].split("\n"):
        print("   " + ln)
    print()

Art. 18, comma 1, lett. c-bis, Codice del Consumo   (letter with nested numbered sub-points)
BEFORE (14 chars):
   c-bis) "beni":

AFTER (506 chars):
   c-bis) "beni":
   1) qualsiasi bene mobile materiale anche da assemblare; l'acqua, il gas e l'energia elettrica quando sono confezionati per la vendita in un volume delimitato o in quantità determinata;
   2) qualsiasi bene mobile materiale che incorpora, o è interconnesso con, un contenuto digitale o un servizio digitale in modo tale che la mancanza di detto contenuto digitale o servizio digitale impedirebbe lo svolgimento delle funzioni proprie del bene ("beni con elementi digitali");
   3) gli animali vivi;

Art. 18, comma 1, lett. n-septies, Codice del Consumo   (nested sub-points AND a cross-reference)
BEFORE (345 chars):
   n-septies) "sistema di certificazione": un sistema di verifica da parte di terzi che certifica che un prodotto, un processo o un'impresa è conforme a determinati requisiti, che consente l'uso di una corrispond

**Art. 18 lett. c-bis** collapsed to the 14 characters `c-bis) "beni":` — the whole
definition gone, leaving a chunk that still embeds and still retrieves.

**Art. 18 lett. n-septies** is worse than truncation. Its `((57))` marker sits at the
end of the *last* sub-point, so cutting at the first line lost the marker too and
filed a provision **inserted by D.Lgs. 30/2026 into the control arm** — a silent
treatment misclassification, exactly the error that would bias a before/after study
and never show up as a crash.

Fixing this moved the treated arm from 24 provisions to **26**.

### A fourth case: the letter that is not a rule

Art. 21 comma 1's letters are not self-contained prohibitions. They are *elements*
of one test — a practice misleads "as to" one of them — unlike Art. 23's letters,
which are per-se fact patterns. Retrieved bare, `lett. b)` is a noun phrase with no
legal operator in it at all.

`context_text` carries the comma's chapeau with every letter, and `embed_text`
(chapeau + text) is what gets embedded. `text` stays verbatim and alone, because it
is what a citation displays.

In [8]:
c = by["cdc-art21-c1-lett-b"]
print("text (verbatim, what a citation shows) — note it is a fragment:\n")
print("   " + c["text"][:300])
print("\n\ncontext_text (the chapeau it belongs to):\n")
print("   " + c["context_text"][:340])
print("\n\nembed_text (what is actually indexed) — now a complete rule:\n")
print("   " + c["embed_text"][:520])

text (verbatim, what a citation shows) — note it is a fragment:

   b) le caratteristiche principali del prodotto, quali la sua disponibilità, i vantaggi, i rischi, l'esecuzione, la composizione, le caratteristiche ambientali o sociali, gli accessori, gli aspetti relativi alla circolarità, quali la durabilità, la riparabilità o la riciclabilità, l'assistenza post-ve


context_text (the chapeau it belongs to):

   1. È considerata ingannevole una pratica commerciale che contiene informazioni non rispondenti al vero o, seppure di fatto corretta, in qualsiasi modo, anche nella sua presentazione complessiva, induce o è idonea ad indurre in errore il consumatore medio riguardo ad uno o più dei seguenti elementi e, in ogni caso, lo induce o è idonea a i


embed_text (what is actually indexed) — now a complete rule:

   1. È considerata ingannevole una pratica commerciale che contiene informazioni non rispondenti al vero o, seppure di fatto corretta, in qualsiasi modo, anche nella sua present

In [9]:
# how many chunks would be fragments without their chapeau?
frag = df[(df.context_text.str.len() > 0)]
print("chunks carrying a chapeau: %d of %d" % (len(frag), len(df)))
print()
print("mean embed_text / text length ratio for those: %.1fx"
      % (frag.embed_text.str.len() / frag.text.str.len()).mean())

chunks carrying a chapeau: 88 of 106

mean embed_text / text length ratio for those: 1.7x


### A fifth: a long general-clause comma

Art. 22 comma 5-ter is the single new Art. 22 provision — comparison services and
environmental characteristics. It is one long sentence and is *not* split: breaking
a materiality test into sentences would produce fragments that each retrieve on
their own and none of which state the rule.

In [10]:
c = by["cdc-art22-c5-ter"]
print(c["citation_it"], "|", c["regime"], "|", len(c["text"]), "chars")
print()
print(c["text"])

Art. 22, comma 5-ter, Codice del Consumo | new-2026 | 508 chars

5-ter. Quando il professionista fornisce un servizio di raffronto fra prodotti e comunica al consumatore informazioni sulle caratteristiche ambientali o sociali o sugli aspetti relativi alla circolarità, quali la durabilità, la riparabilità o la riciclabilità, dei prodotti o dei fornitori di tali prodotti, sono considerate rilevanti le informazioni sul metodo di raffronto, sui prodotti raffrontati e sui fornitori di tali prodotti, così come sulle misure predisposte per tenere aggiornate le informazioni.


## 4. Definitional dependencies

The reason a blacklist letter cannot be retrieved alone. Art. 23 lett. d-bis reads:

> *formulare un'**asserzione ambientale generica** per la quale il professionista non
> è in grado di dimostrare l'**eccellenza riconosciuta delle prestazioni ambientali**
> pertinenti all'asserzione*

Both bolded phrases are terms of art defined in Art. 18. Without them the rule is
unusable — an LLM asked "does this claim match?" has nothing to match against.
`depends_on` is resolved by finding defined terms verbatim in the rule's text,
longest term first so *asserzione ambientale generica* claims the span before
*asserzione ambientale* can.

In [11]:
dep = df[df.depends_on.str.len() > 0]
print("chunks with at least one definitional dependency: %d" % len(dep))
print("most depended-upon definitions:\n")
cnt = collections.Counter(d for ds in df.depends_on for d in ds)
pd.DataFrame([{"definition": by[k]["citation_it"],
               "term": by[k]["defined_term"], "depended on by": v}
              for k, v in cnt.most_common(10)])

chunks with at least one definitional dependency: 63
most depended-upon definitions:



,definition,term,depended on by
0,"Art. 18, comma 1, lett. a, Codice del Consumo",consumatore,30
1,"Art. 18, comma 1, lett. c, Codice del Consumo",prodotto,28
2,"Art. 18, comma 1, lett. b, Codice del Consumo",professionista,23
3,"Art. 18, comma 1, lett. m, Codice del Consumo",decisione di natura commerciale,14
4,"Art. 18, comma 1, lett. i, Codice del Consumo",invito all'acquisto,6
5,"Art. 18, comma 1, lett. n-novies, Codice del Consumo",durabilità,4
6,"Art. 18, comma 1, lett. h, Codice del Consumo",diligenza professionale,2
7,"Art. 18, comma 1, lett. n-quater, Codice del Consumo",asserzione ambientale,2
8,"Art. 18, comma 1, lett. c-bis, Codice del Consumo",beni,2
9,"Art. 18, comma 1, lett. n-bis, Codice del Consumo",classificazione,2


In [12]:
for cid in ["cdc-art23-c1-lett-b-bis", "cdc-art23-c1-lett-d-bis",
            "cdc-art23-c1-lett-d-ter", "cdc-art23-c1-lett-d-quater"]:
    c = by[cid]
    print("%s   [%s]" % (c["citation_it"], c["regime"]))
    print("   " + c["text"][:210])
    for d in c["depends_on"]:
        print("      needs -> %-42s (%s)" % (by[d]["defined_term"], by[d]["citation_it"]))
    print()

Art. 23, comma 1, lett. b-bis, Codice del Consumo   [new-2026]
   b-bis) esibire una etichetta di sostenibilità che non è basata su un sistema di certificazione o non è stabilita da autorità pubbliche;
      needs -> sistema di certificazione                  (Art. 18, comma 1, lett. n-septies, Codice del Consumo)
      needs -> etichetta di sostenibilità                 (Art. 18, comma 1, lett. n-sexies, Codice del Consumo)

Art. 23, comma 1, lett. d-bis, Codice del Consumo   [new-2026]
   d-bis) formulare un'asserzione ambientale generica per la quale il professionista non è in grado di dimostrare l'eccellenza riconosciuta delle prestazioni ambientali pertinenti all'asserzione;
      needs -> professionista                             (Art. 18, comma 1, lett. b, Codice del Consumo)
      needs -> eccellenza riconosciuta delle prestazioni ambientali (Art. 18, comma 1, lett. n-octies, Codice del Consumo)
      needs -> asserzione ambientale generica             (Art. 18, comma 1, lett.

## 5. Known defect in the source

Carried, not repaired.

In [13]:
bad = df[df.source_defect.notna()]
for r in bad.itertuples():
    print("%s  ->  %s" % (r.citation_it, r.source_defect))
    print("   verbatim: " + r.text[:110])
print()
print("pdftotext and pdfplumber agree on this rendering, so it is in the PDF.")
print("`text` stays verbatim; `defined_term` is null rather than guessed.")
print("No impact on the study: 'microimpresa' scopes B2B protection, not product claims.")

Art. 18, comma 1, lett. d-bis, Codice del Consumo  ->  mangled quotation marks around the defined term
   verbatim: d-bis) 'microimpresé: entità, società o associazioni che, a prescindere dalla forma giuridica, esercitano un'a

pdftotext and pdfplumber agree on this rendering, so it is in the PDF.
`text` stays verbatim; `defined_term` is null rather than guessed.
No impact on the study: 'microimpresa' scopes B2B protection, not product claims.


## 6. Full chunk table

Everything in the index, readable.

In [14]:
view = df[["chunk_id", "citation_it", "chunk_type", "regime", "defined_term"]].copy()
view["text (truncated)"] = df.text.str.replace("\n", " ", regex=False).str.slice(0, 115)
view["deps"] = df.depends_on.str.len()
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 118):
    display(view.set_index("chunk_id"))

,citation_it,chunk_type,regime,defined_term,text (truncated),deps
chunk_id,,,,,,
cdc-art18-c1-lett-a,"Art. 18, comma 1, lett. a, Codice del Consumo",definition,pre-existing,consumatore,"a) ""consumatore"": qualsiasi persona fisica che, nelle pratiche commerciali oggetto del presente titolo, agisce per",0
cdc-art18-c1-lett-b,"Art. 18, comma 1, lett. b, Codice del Consumo",definition,pre-existing,professionista,"b) ""professionista"": qualsiasi persona fisica o giuridica che, nelle pratiche commerciali oggetto del presente tito",0
cdc-art18-c1-lett-c,"Art. 18, comma 1, lett. c, Codice del Consumo",definition,pre-existing,prodotto,"c) ""prodotto"": qualsiasi bene o servizio, compresi i beni immobili, i servizi digitali e il contenuto digitale, non",0
cdc-art18-c1-lett-c-bis,"Art. 18, comma 1, lett. c-bis, Codice del Consumo",definition,new-2026,beni,"c-bis) ""beni"": 1) qualsiasi bene mobile materiale anche da assemblare; l'acqua, il gas e l'energia elettrica quando",0
cdc-art18-c1-lett-d,"Art. 18, comma 1, lett. d, Codice del Consumo",definition,pre-existing,pratiche commerciali tra professionisti e consumatori,"d) ""pratiche commerciali tra professionisti e consumatori"" (di seguito denominate: ""pratiche commerciali""): qualsia",0
cdc-art18-c1-lett-d-bis,"Art. 18, comma 1, lett. d-bis, Codice del Consumo",definition,pre-existing,None,"d-bis) 'microimpresé: entità, società o associazioni che, a prescindere dalla forma giuridica, esercitano un'attivi",0
cdc-art18-c1-lett-e,"Art. 18, comma 1, lett. e, Codice del Consumo",definition,pre-existing,falsare in misura rilevante il comportamento economico dei consumatori,"e) ""falsare in misura rilevante il comportamento economico dei consumatori"": l'impiego di una pratica commerciale i",0
cdc-art18-c1-lett-f,"Art. 18, comma 1, lett. f, Codice del Consumo",definition,pre-existing,codice di condotta,"f) ""codice di condotta"": un accordo o una normativa che non è imposta dalle disposizioni legislative, regolamentari",0
cdc-art18-c1-lett-g,"Art. 18, comma 1, lett. g, Codice del Consumo",definition,pre-existing,responsabile del codice,"g) ""responsabile del codice"": qualsiasi soggetto, compresi un professionista o un gruppo di professionisti, respons",0


## 7. Self-sufficiency check

A chunk that mentions a defined term it does not link, or that is too short to state
a rule, is a chunk the LLM will misread. Both are asserted here rather than eyeballed.

In [15]:
short = df[(df.chunk_type != "definition") & (df.embed_text.str.len() < 120)]
print("operative chunks under 120 chars after adding context:", len(short))
if len(short):
    print(short[["citation_it", "embed_text"]].to_string())

terms = {c["defined_term"].lower(): c["chunk_id"] for c in cs if c.get("defined_term")}
unlinked = []
for c in cs:
    if c["chunk_type"] == "definition":
        continue
    for t_, cid in terms.items():
        if t_ in c["embed_text"].lower() and cid not in c["depends_on"]:
            covered = any(t_ in by[d]["defined_term"].lower() for d in c["depends_on"])
            if not covered:
                unlinked.append((c["chunk_id"], t_))
print("\nunlinked defined-term mentions:", len(unlinked))
for u in unlinked[:10]:
    print("   ", u)

operative chunks under 120 chars after adding context: 3
                                      citation_it                                                                                                    embed_text
32           Art. 20, comma 1, Codice del Consumo                                                            1. Le pratiche commerciali scorrette sono vietate.
35  Art. 20, comma 4, lett. a, Codice del Consumo  4. In particolare, sono scorrette le pratiche commerciali: a) ingannevoli di cui agli articoli 21, 22 e 23 o
36  Art. 20, comma 4, lett. b, Codice del Consumo    4. In particolare, sono scorrette le pratiche commerciali: b) aggressive di cui agli articoli 24, 25 e 26.

unlinked defined-term mentions: 0


In [16]:
# citation identity must be unique -- Art. 21 carries a lett. b) in both commas
ids = collections.Counter((c["article"], c["comma"], c["letter"]) for c in cs)
assert not [k for k, v in ids.items() if v > 1], "ambiguous citation identity"
print("citation identity unique across all %d chunks" % len(cs))
both = [c["citation_it"] for c in cs if c["article"] == "21" and c["letter"] == "b"]
print("\nthe reason the comma is load-bearing:")
for b in both:
    print("   " + b)

citation identity unique across all 106 chunks

the reason the comma is load-bearing:
   Art. 21, comma 1, lett. b, Codice del Consumo
   Art. 21, comma 2, lett. b, Codice del Consumo


## 8. Gate summary

**106 chunks** over Art. 18–23: 25 definitions, 7 scope-limit, 31 general-clause,
39 blacklist. Verification (char_span reproduces text, unique ids, no dangling
dependencies, coherent treated arm) passes with no problems.

**The treated arm is 26 provisions, not 24.** Two were misfiled as pre-existing
because their amendment marker sits at the end of a nested sub-point that the first
splitter discarded (Art. 18 lett. c-bis and lett. n-septies). That is the failure
mode worth remembering: it produced no error, no empty field and no crash — just a
quietly wrong control/treatment split, which is precisely what a before/after study
cannot survive.

**Decisions taken, open to reversal**

1. **Art. 19 added** to scope, as `scope-limit`. It is the rule that says this title
   may be displaced by sector-specific EU law — load-bearing for food health and
   nutrition claims.
2. **`scope-limit` added** to Appendix B's `chunk_type` enum. Art. 19 is neither
   definition, blacklist nor general clause, and should surface *alongside* a match
   to qualify it, never as the primary answer.
3. **Sub-points are not separate chunks.** They are parts of their letter's fact
   pattern, not independently citable.
4. **Chapeaux are carried, not merged.** `text` stays verbatim and citable;
   `embed_text` = chapeau + text is what gets indexed.
5. **Art. 24–26 and Art. 45/48/49/65-ter excluded**, for the reasons in §1.

**Next**, once these chunks look right: embed `embed_text`, and check whether
retrieval surfaces the correct *provision family* for a claim — measured as
in-scope / not-in-scope, which is what the study actually needs, rather than
exact-letter top-1.